In [2]:
import pandas as pd

## Sensors

In [4]:
min_list=[]
max_list=[]
for i in range(1, 36+1):
    dfSensor = pd.read_excel('../data/DraginoSoilMositure_Morocco_Season1.xlsx', sheet_name=f'Sensor {i}',
                             usecols=[0, 1])
    dfSensor.rename(columns={"Row Labels": "datetime", "Average of water_SOIL": "sm_value"}, inplace=True)
    dfSensor["datetime"] = pd.to_datetime(dfSensor["datetime"])
    min_list.append(dfSensor["datetime"].min())
    max_list.append(dfSensor["datetime"].max())
print(f"Min: {min_list}")
print(f"Max: {max_list}")
max_abs=min(max_list)
min_abs=max(min_list)
print(min_abs)
print(max_abs)
full_index = pd.date_range(min_abs, max_abs, freq="h")

Min: [Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestamp('2024-12-12 12:00:00'), Timestam

In [7]:
sensors = {}
for i in range(1, 36+1):
    dfSensor = pd.read_excel('../data/DraginoSoilMositure_Morocco_Season1.xlsx', sheet_name=f'Sensor {i}',
                             usecols=[0, 1])
    dfSensor.rename(columns={"Row Labels": "datetime", "Average of water_SOIL": "sm_value"}, inplace=True)

    dfSensor["datetime"] = pd.to_datetime(dfSensor["datetime"])
    dfSensor.set_index("datetime", inplace=True)
    dfSensor = dfSensor.reindex(full_index)
    dfSensor["sm_value"] = dfSensor["sm_value"].interpolate()
    sensors[i] = dfSensor


## Meteo

In [8]:
dfMeteo = pd.read_csv('../data/open-meteo.csv')
dfMeteo["datetime"] = pd.to_datetime(dfMeteo["datetime"])

dfMeteo.set_index("datetime", inplace=True)
dfMeteo = dfMeteo.reindex(full_index)
print(dfMeteo.index.min())
print(dfMeteo.index.max())
print(dfMeteo.shape)

2024-12-12 12:00:00
2025-05-15 12:00:00
(3697, 11)


In [9]:
dfMeteo_F = pd.read_csv('../data/open-meteo-Forecast.csv')
dfMeteo_F["datetime"] = pd.to_datetime(dfMeteo_F["datetime"])

dfMeteo_F.set_index("datetime", inplace=True)
dfMeteo_F = dfMeteo_F.reindex(full_index)
print(dfMeteo_F.index.min())
print(dfMeteo_F.index.max())
print(dfMeteo_F.shape)

2024-12-12 12:00:00
2025-05-15 12:00:00
(3697, 9)


In [ ]:
dfMeteoStation = pd.read_excel('WeatherStation.xlsx', usecols=[1,2,3,4,5,6,7,8,9])
print(dfMeteoStation)
dfMeteoStation["datetime"] = dfMeteoStation["datetime"].dt.round('h')
dfMeteoStation_rev = dfMeteoStation.groupby('datetime').mean()

dfMeteoStation_rev = dfMeteoStation_rev.reindex(full_index)



## LAI

In [10]:
dfLAI = pd.read_excel('../data/LAI_Morocco_Season1.xlsx', sheet_name=f'Sheet1')
dfLAI["datetime"] = pd.to_datetime(dfLAI["datetime"]) + pd.to_timedelta("12:00:00")
dfLAI = dfLAI.set_index("datetime")
dfLAI = dfLAI.reindex(full_index)

values = {
    "Tititcaca-CROP": 0,
    "Tititcaca-Sensor": 0,
    "Tititcaca-Farmer": 0,
    "ICBA-CROP": 0,
    "ICBA-Sensor": 0,
    "ICBA-Farmer": 0
}
dfLAI.loc["2024-12-12 12:00:00"] = values

dfLAI[['Tititcaca-CROP', 'Tititcaca-Sensor', 'Tititcaca-Farmer', 'ICBA-CROP',
       'ICBA-Sensor', 'ICBA-Farmer']] = dfLAI[['Tititcaca-CROP', 'Tititcaca-Sensor', 'Tititcaca-Farmer', 'ICBA-CROP',
                                               'ICBA-Sensor', 'ICBA-Farmer']].interpolate()

In [11]:

mapToSensor = {'Tititcaca-CROP': [5, 6, 23, 24, 31, 32],
               'Tititcaca-Sensor': [3, 4, 19, 20, 35, 36],
               'Tititcaca-Farmer': [9, 10, 13, 14, 29, 30],
               'ICBA-CROP': [1, 2, 15, 16, 33, 34],
               'ICBA-Sensor': [7, 8, 21, 22, 25, 26],
               'ICBA-Farmer': [11, 12, 17, 18, 27, 28]}

for i in range(1, 37):
    for k, v in mapToSensor.items():
        if i in v:
            dfLAI[f'{i}'] = dfLAI[k]

dfLAI.drop(['Tititcaca-CROP', 'Tititcaca-Sensor', 'Tititcaca-Farmer', 'ICBA-CROP',
            'ICBA-Sensor', 'ICBA-Farmer'], axis=1, inplace=True)


# Irrigation

In [16]:
dfIRR = pd.DataFrame(columns=['f1','f2','f3','f4','f5','f6','f7','f8','f9','f10','f11','f12','f13','f14','f15','f16','f17','f18'], index=full_index)
dfIRR.fillna(0.0, inplace=True)

/var/folders/66/rpqcrym93h90yphhq7kgwhb00000gn/T/ipykernel_88433/450902979.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dfIRR.fillna(0.0, inplace=True)


In [17]:
dfIrrigation = pd.read_excel('../data/IrrigationEvents_Morocco_Season1.xlsx', sheet_name='Irrigation')
dfIrrigation.rename(columns={"Date": "datetime", "Irrigation Duration": "duration", "Irrigation Volume": "volume"},
                     inplace=True)
dfIrrigation.drop(['Soil Mositure before irrigation (only Sensor)','duration'], axis=1, inplace=True)
dfIrrigation["datetime"] = pd.to_datetime(dfIrrigation["datetime"]) + pd.to_timedelta("12:00:00")
dfIrrigation.sort_values("datetime", ascending=True, inplace=True)
CW = ['f1', 'f3', 'f8', 'f12', 'f16','f17']
F = ['f5', 'f6', 'f7', 'f9', 'f14', 'f15']
#S = ['f2', 'f4', 'f10', 'f11', 'f13','f18']
dfIrrigation["Plot"] = dfIrrigation["Plot"].apply(
    lambda x: CW if x == "CROPWAT" else F if x == "Farmer" else f"f{x}"
)
dfIrrigation.head()

,datetime,Plot,volume
0,2025-02-22 12:00:00,"[f1, f3, f8, f12, f16, f17]",18.500000
20,2025-02-24 12:00:00,f13,16.352201
19,2025-02-24 12:00:00,f11,19.496855
18,2025-02-24 12:00:00,f10,23.270440
17,2025-02-24 12:00:00,f4,18.867925


In [18]:
for irr_ev in dfIrrigation.iterrows():
    #print(irr_ev[1]['Plot'])
    for p in irr_ev[1]['Plot']:
        dfIRR.loc[pd.to_datetime(irr_ev[1]['datetime']), p] = irr_ev[1]['volume']
        #print(dfIRR.loc[irr_ev[1]['datetime'], p])
        #pd.to_datetime(irr_ev[1]['datetime']),

In [19]:
print(sensors[1].shape, sensors[1].columns)
print(dfIRR.shape, dfIRR.columns)
print(dfMeteo.shape, dfMeteo.columns)

(3697, 1) Index(['sm_value'], dtype='object')
(3697, 25) Index(['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11',
       'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f', '1', '3', '0',
       '4', '2', '8'],
      dtype='object')
(3697, 11) Index(['temperature_2m', 'relative_humidity_2m', 'cloud_cover',
       'wind_speed_10m', 'wind_direction_100m', 'soil_temperature_0_to_7cm',
       'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 'rain',
       'precipitation', 'evapotranspiration'],
      dtype='object')


In [20]:
meteo_fields = ['temperature_2m', 'relative_humidity_2m', 'cloud_cover',
       'wind_speed_10m', 'wind_direction_100m', 'soil_temperature_0_to_7cm',
       'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 'rain',
       'precipitation', 'evapotranspiration']
meteo_fields_forecast = ['temperature_2m_f','relative_humidity_2m_f','cloud_cover_f','wind_speed_10m_f','wind_direction_10m_f','soil_temperature_0cm_f','soil_temperature_6cm_f','soil_temperature_18cm_f','rain_f']
all_fields = []

for i, field in enumerate(['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11',
       'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18'], start=1):
    df_new = pd.DataFrame(columns=['s_a','s_b','irr','temperature_2m', 'relative_humidity_2m', 'cloud_cover',
       'wind_speed_10m', 'wind_direction_100m', 'soil_temperature_0_to_7cm',
       'soil_temperature_7_to_28cm', 'soil_temperature_28_to_100cm', 'rain',
       'precipitation', 'evapotranspiration'], index=full_index)
    df_new['irr'] = dfIRR[field]
    df_new['s_a'] = sensors[i*2]['sm_value']
    df_new['s_b'] = sensors[i*2-1]['sm_value']
    df_new['LAI'] = dfLAI[f'{i}']
    df_new[meteo_fields] = dfMeteo[meteo_fields]
    df_new[meteo_fields_forecast] = dfMeteo_F[meteo_fields_forecast]
    df_new.to_csv(f'fieldsComplete/field_{field}',index_label='datetime')


In [21]:
df = pd.read_csv('fieldsComplete/field_f1')
df.head()

,datetime,s_a,s_b,irr,temperature_2m,relative_humidity_2m,cloud_cover,wind_speed_10m,wind_direction_100m,soil_temperature_0_to_7cm,...,LAI,temperature_2m_f,relative_humidity_2m_f,cloud_cover_f,wind_speed_10m_f,wind_direction_10m_f,soil_temperature_0cm_f,soil_temperature_6cm_f,soil_temperature_18cm_f,rain_f
0,2024-12-12 12:00:00,13.43,12.69,0.0,18.1,35,0,6.4,311,14.3,...,0.000000,16.2,44,27,3.0,284,19.4,12.9,10.2,0.0
1,2024-12-12 13:00:00,13.45,12.69,0.0,18.9,34,0,7.1,334,15.1,...,0.000541,17.5,42,41,2.3,321,20.0,14.4,10.6,0.0
2,2024-12-12 14:00:00,13.55,12.68,0.0,18.9,33,1,7.0,351,16.0,...,0.001082,18.2,41,20,2.5,352,20.0,15.4,11.1,0.0
3,2024-12-12 15:00:00,13.56,12.69,0.0,17.8,41,1,9.6,40,16.2,...,0.001623,17.9,45,35,7.9,51,17.9,16.0,11.6,0.0
4,2024-12-12 16:00:00,13.62,12.72,0.0,15.9,50,0,8.4,74,15.7,...,0.002164,16.9,52,2,9.4,72,15.0,15.5,12.0,0.0
